# 03 — Preprocessing and Model Training

**Purpose.** Build the two parallel corpora that the whole study rests on, then
train both architectures on each of them.

The experimental design is a **controlled two-track ablation**. One preprocessing
pipeline is built and forked at exactly one operation:

* **Track A — `text_no_emoji`** — every emoji is deleted.
* **Track B — `text_with_emoji`** — every emoji is replaced by its Unicode description.

Upstream and downstream of that fork both tracks are treated identically, so the
only systematic difference between the two corpora is whether emoji content
survives as text. Each track then trains both architectures under an identical
protocol, giving four comparable conditions.

| | |
|---|---|
| **Input**  | `data/interim/{train,dev,test}_raw.csv` |
| **Output** | `data/processed/*_cleaned.csv`, `models/*.keras`, `results/thresholds.json`, `results/best_configs.json`, `results/hyperparameter_search.csv` |
| **Next**   | `04_evaluation.ipynb` |

> **Note.** The attention model is a Transformer encoder trained **from scratch**.
> It is stored as `bert_*.keras` for continuity with the original notebook, but it
> is not pretrained BERT and should not be described as such.

In [ ]:
# emoji  -> Track A (deletion)   demoji -> Track B (verbalisation)
try:
    import emoji, demoji
except ImportError:
    !pip install -q emoji demoji
    import emoji, demoji

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_KAGGLE:
    PROJECT = Path("/kaggle/working")
    RAW_CANDIDATES = [Path("/kaggle/input")]
else:
    # notebooks/ lives one level below the project root
    PROJECT = Path.cwd()
    if PROJECT.name == "notebooks":
        PROJECT = PROJECT.parent
    RAW_CANDIDATES = [PROJECT / "data" / "raw"]

INTERIM   = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
MODELS    = PROJECT / "models"
RESULTS   = PROJECT / "results"
FIGURES   = PROJECT / "figures"
for d in (INTERIM, PROCESSED, MODELS, RESULTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

EMOTION_LABELS = [
    "anger", "anticipation", "disgust", "fear", "joy", "love",
    "optimism", "pessimism", "sadness", "surprise", "trust",
]
TRACKS = [
    ("no_emoji",   "text_no_emoji",   "Without Emoji"),
    ("with_emoji", "text_with_emoji", "With Emoji"),
]

print("project root :", PROJECT)
print("on kaggle    :", ON_KAGGLE)

In [ ]:
import json, re, string
import numpy as np
import pandas as pd
import tensorflow as tf
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import f1_score

for pkg in ("punkt", "wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)

MAX_TOKENS      = 20000
SEQUENCE_LENGTH = 128
SEARCH_EPOCHS   = 5
FINAL_EPOCHS    = 10

## 3.1 The preprocessing pipeline

Five stages. Stage 2 is the fork and is the **only** operation that differs
between the tracks.

> These functions are the single source of truth for what the models were trained
> on. `app/preprocessing.py` is a deliberate copy for serving — if you change
> anything here, change it there too, or every prediction the UI makes becomes
> invalid.

In [ ]:
tweet_tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
lemmatizer = WordNetLemmatizer()

slang_map = {
    "u": "you", "ur": "your", "r": "are", "lol": "laugh", "omg": "surprise",
    "im": "i am", "cant": "cannot", "dont": "do not", "gonna": "going to",
}


def clean_text(text):
    """Stage 1 - strip URLs, @mentions and the # symbol (keeping the word)."""
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()


def handle_emoji_no(text):
    """Stage 2a - TRACK A: delete every emoji."""
    return emoji.replace_emoji(str(text), replace="")


def handle_emoji_yes(text):
    """Stage 2b - TRACK B: replace each emoji with its Unicode description."""
    return demoji.replace_with_desc(str(text), sep=" ")


def normalize_text(text):
    """Stage 3 - lowercase, strip non-alphanumerics."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def tokenize_text(text):
    """Stage 4 - TweetTokenizer, dropping bare punctuation."""
    return [t for t in tweet_tokenizer.tokenize(text) if t not in string.punctuation]


def lemmatize_tokens(tokens):
    """Stage 5 - slang expansion then WordNet lemmatisation."""
    tokens = [slang_map.get(t, t) for t in tokens]
    return [lemmatizer.lemmatize(t) for t in tokens]


def build_text(text, emoji_mode="no"):
    """Full pipeline. emoji_mode='no' -> Track A, 'yes' -> Track B."""
    text = clean_text(text)
    text = handle_emoji_no(text) if emoji_mode == "no" else handle_emoji_yes(text)
    text = normalize_text(text)
    return " ".join(lemmatize_tokens(tokenize_text(text)))

### Worked example — where the two tracks diverge

Run on a constructed sentence rather than a real tweet, so no corpus text is
reproduced.

In [ ]:
sample = "I'm so happy today \U0001F602\U0001F62D u cant believe it!! #joy @friend http://x.com"

s1 = clean_text(sample)
a2, b2 = handle_emoji_no(s1), handle_emoji_yes(s1)
a3, b3 = normalize_text(a2), normalize_text(b2)
a5 = " ".join(lemmatize_tokens(tokenize_text(a3)))
b5 = " ".join(lemmatize_tokens(tokenize_text(b3)))

stages = pd.DataFrame([
    {"stage": "0. input",                    "Track A": sample, "Track B": sample},
    {"stage": "1. noise removal",            "Track A": s1,  "Track B": s1},
    {"stage": "2. emoji handling  <- FORK",  "Track A": a2,  "Track B": b2},
    {"stage": "3. normalisation",            "Track A": a3,  "Track B": b3},
    {"stage": "5. slang + lemmatisation",    "Track A": a5,  "Track B": b5},
])
pd.set_option("display.max_colwidth", 100)
display(stages)

print("\nTrack B recovered these extra tokens from the emoji:")
print(" ", sorted(set(b5.split()) - set(a5.split())))

## 3.2 Apply the pipeline to every split

In [ ]:
splits = {s: pd.read_csv(INTERIM / f"{s}_raw.csv") for s in ("train", "dev", "test")}

for name, df in splits.items():
    df["text_no_emoji"]   = df["text"].apply(lambda t: build_text(t, "no"))
    df["text_with_emoji"] = df["text"].apply(lambda t: build_text(t, "yes"))
    print(f"{name:6s} preprocessed  ({len(df)} rows)")

train_df, dev_df, test_df = splits["train"], splits["dev"], splits["test"]

### Did preprocessing damage anything?

Two failure modes matter: rows reduced to an empty string (the model would receive
nothing), and the two tracks accidentally coming out identical (which would mean
the fork did nothing).

In [ ]:
check = []
for name, df in splits.items():
    a, b = df["text_no_emoji"], df["text_with_emoji"]
    check.append({
        "split": name,
        "mean length A": round(a.str.len().mean(), 1),
        "mean length B": round(b.str.len().mean(), 1),
        "empty A": int((a.str.strip() == "").sum()),
        "empty B": int((b.str.strip() == "").sum()),
        "rows where tracks differ": int((a != b).sum()),
        "% differing": round(100 * (a != b).mean(), 1),
    })
quality = pd.DataFrame(check)
display(quality)

print("The '% differing' column is the share of rows the manipulation can act on;")
print("it should track the emoji density measured in notebook 02.")

# Rows that became empty are almost always emoji-only tweets stripped by Track A.
empty_a = train_df[train_df["text_no_emoji"].str.strip() == ""]
if len(empty_a):
    print(f"\n{len(empty_a)} training rows are empty on Track A (emoji-only tweets).")
    display(empty_a[["text", "text_with_emoji"]].head(3))

In [ ]:
SAVE_COLS = ["text", "text_no_emoji", "text_with_emoji"] + EMOTION_LABELS

for name, df in splits.items():
    out = PROCESSED / f"{name}_cleaned.csv"
    df[SAVE_COLS].to_csv(out, index=False, encoding="utf-8")
    print("saved", out)

# A small fixture so the serving copy of the pipeline can be checked for drift.
fixture = {"cases": [{"input": s,
                      "track_a": build_text(s, "no"),
                      "track_b": build_text(s, "yes")}
                     for s in [sample,
                               "waiting to hear back \U0001F630 fingers crossed",
                               "absolutely gutted about the result today"]]}
(RESULTS / "pipeline_fixture.json").write_text(
    json.dumps(fixture, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved", RESULTS / "pipeline_fixture.json")

## 3.3 Class imbalance

Notebook 02 established that positive instances differ by roughly an order of
magnitude across labels. An unweighted objective is minimised most efficiently by
predicting the negative class for rare emotions, so each label is weighted by its
own negative-to-positive ratio.

In [ ]:
pos_counts = train_df[EMOTION_LABELS].sum().values.astype(np.float32)
pos_weight_values = (len(train_df) - pos_counts) / np.maximum(pos_counts, 1.0)
POS_WEIGHT = tf.constant(pos_weight_values, dtype=tf.float32)

display(pd.DataFrame({"emotion": EMOTION_LABELS,
                      "positives": pos_counts.astype(int),
                      "pos_weight": pos_weight_values.round(2)})
          .sort_values("pos_weight", ascending=False))


def weighted_bce(y_true, y_pred):
    """Binary cross-entropy on logits, weighted per label."""
    return tf.reduce_mean(tf.nn.weighted_cross_entropy_with_logits(
        labels=y_true, logits=y_pred, pos_weight=POS_WEIGHT))

## 3.4 Data pipeline and model builders

In [ ]:
def make_vectorizer(train_texts, max_tokens=MAX_TOKENS):
    """Adapted per track, so Track B's vocabulary includes description words."""
    vec = tf.keras.layers.TextVectorization(
        max_tokens=max_tokens, output_sequence_length=SEQUENCE_LENGTH,
        standardize=None)          # normalisation already done in the pipeline
    vec.adapt(train_texts.astype(str).values)
    return vec, len(vec.get_vocabulary())


def make_dataset(df, text_col, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        df[text_col].astype(str).values,
        df[EMOTION_LABELS].values.astype(np.float32)))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_lstm(vectorizer, vocab_size, p):
    """Recurrent arm: two stacked BiLSTMs, tapering to the classification head."""
    inp = tf.keras.Input(shape=(), dtype=tf.string)
    x = vectorizer(inp)
    x = tf.keras.layers.Embedding(vocab_size, p["embed_dim"], mask_zero=True)(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(p["lstm_units"], return_sequences=True,
                             dropout=p["dropout"]))(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(p["lstm_units"] // 2, dropout=p["dropout"]))(x)
    x = tf.keras.layers.Dropout(p["dropout"])(x)
    out = tf.keras.layers.Dense(len(EMOTION_LABELS))(x)   # raw logits
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(p["lr"]), loss=weighted_bce)
    return m


def build_transformer(vectorizer, vocab_size, p):
    """Attention arm: one Transformer encoder block, trained from scratch."""
    inp = tf.keras.Input(shape=(), dtype=tf.string)
    x = vectorizer(inp)
    x = tf.keras.layers.Embedding(vocab_size, p["embed_dim"], mask_zero=True)(x)
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=p["num_heads"], key_dim=p["key_dim"])(x, x)
    x = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([x, attn]))
    ff = tf.keras.layers.Dense(p["ff_dim"], activation="relu")(x)
    ff = tf.keras.layers.Dropout(p["dropout"])(ff)
    x = tf.keras.layers.LayerNormalization()(
        tf.keras.layers.Add()([x, tf.keras.layers.Dense(p["embed_dim"])(ff)]))
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(p["dropout"])(x)
    out = tf.keras.layers.Dense(len(EMOTION_LABELS))(x)   # raw logits
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(p["lr"]), loss=weighted_bce)
    return m

## 3.5 Prediction and threshold selection

The models emit **raw logits**, so the sigmoid is applied here. Keeping the
conventional 0.5 cut would systematically disadvantage rare labels under a
weighted loss, so the threshold is swept on the development split and the value
maximising Micro-F1 is adopted — applied identically in all four conditions, so it
cannot favour one track over the other.

In [ ]:
def predict(model, ds, label_df, threshold=0.5):
    probs = tf.nn.sigmoid(model.predict(ds, verbose=0)).numpy()
    preds = (probs >= threshold).astype(int)
    return label_df[EMOTION_LABELS].values.astype(int), preds, probs


def best_threshold(y_true, y_prob):
    """Sweep 0.10 -> 0.55 and keep the value that maximises Micro-F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.6, 0.05):
        f1 = f1_score(y_true, (y_prob >= t).astype(int),
                      average="micro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return round(best_t, 2), best_f1

## 3.6 Hyper-parameter grids

In [ ]:
LSTM_PARAM_GRID = [
    {"name": "cfg1", "embed_dim": 128, "lstm_units": 128, "lr": 1e-3, "dropout": 0.2, "batch_size": 64},
    {"name": "cfg2", "embed_dim": 256, "lstm_units": 256, "lr": 5e-4, "dropout": 0.3, "batch_size": 32},
    {"name": "cfg3", "embed_dim": 128, "lstm_units": 64,  "lr": 2e-3, "dropout": 0.1, "batch_size": 64},
]

TRANSFORMER_PARAM_GRID = [
    {"name": "cfg1", "embed_dim": 128, "num_heads": 4, "key_dim": 32, "ff_dim": 256, "lr": 1e-3, "dropout": 0.2, "batch_size": 64},
    {"name": "cfg2", "embed_dim": 256, "num_heads": 8, "key_dim": 32, "ff_dim": 512, "lr": 5e-4, "dropout": 0.3, "batch_size": 32},
    {"name": "cfg3", "embed_dim": 128, "num_heads": 2, "key_dim": 64, "ff_dim": 256, "lr": 2e-3, "dropout": 0.1, "batch_size": 64},
]

# cfg1 is a moderate baseline; cfg2 raises capacity while lowering the learning
# rate; cfg3 reduces capacity while raising it. The same budget is spent on all
# four conditions so the comparison stays fair.
print("configurations per condition:", len(LSTM_PARAM_GRID))

## 3.7 Two-phase training

**Phase 1** trains each configuration briefly and keeps the one with the best
development Micro-F1. **Phase 2** retrains that winner for longer. Early stopping
with best-weight restoration is active in both, so the evaluated model is always
the one at minimum validation loss rather than the one at the final epoch.

In [ ]:
def search(build_fn, param_grid, text_col, epochs):
    """Phase 1 - returns (per-config results, best params)."""
    rows, best_score, best_params = [], -1, None

    for params in param_grid:
        print(f"    {params['name']}: lr={params['lr']} embed={params['embed_dim']} "
              f"batch={params['batch_size']}", flush=True)
        tf.keras.backend.clear_session()
        vec, vocab = make_vectorizer(train_df[text_col])
        model = build_fn(vec, vocab, params)
        model.fit(make_dataset(train_df, text_col, params["batch_size"], shuffle=True),
                  validation_data=make_dataset(dev_df, text_col, params["batch_size"]),
                  epochs=epochs, verbose=0,
                  callbacks=[tf.keras.callbacks.EarlyStopping(
                      monitor="val_loss", patience=2, restore_best_weights=True)])

        val_ds = make_dataset(dev_df, text_col, params["batch_size"])
        y_true, _, y_prob = predict(model, val_ds, dev_df)
        thr, _ = best_threshold(y_true, y_prob)
        _, y_pred, _ = predict(model, val_ds, dev_df, thr)

        micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
        macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
        rows.append({"config": params["name"],
                     "micro_f1": round(micro * 100, 2),
                     "macro_f1": round(macro * 100, 2),
                     "label_accuracy": round((y_true == y_pred).mean() * 100, 2),
                     "threshold": thr})
        print(f"      -> Micro-F1 {micro * 100:.2f}%")

        if micro > best_score:
            best_score, best_params = micro, params

    return pd.DataFrame(rows), best_params

In [ ]:
tuning_log, thresholds, best_configs = [], {}, {}

for model_name, build_fn, grid in [
    ("LSTM", build_lstm, LSTM_PARAM_GRID),
    ("BERT", build_transformer, TRANSFORMER_PARAM_GRID),   # file name kept; see note
]:
    label = "BiLSTM" if model_name == "LSTM" else "Transformer"
    print(f"\n{'=' * 62}\n  {label}\n{'=' * 62}")

    for track_key, text_col, track_label in TRACKS:
        print(f"\n--- {label} | {track_label} ---")

        print("  Phase 1: hyper-parameter search")
        search_df, best_params = search(build_fn, grid, text_col, SEARCH_EPOCHS)
        search_df["model"], search_df["track"] = model_name, track_label
        tuning_log.append(search_df)
        display(search_df)
        print(f"  best config: {best_params['name']}")

        print("  Phase 2: final training")
        tf.keras.backend.clear_session()
        vec, vocab = make_vectorizer(train_df[text_col])
        model = build_fn(vec, vocab, best_params)
        model.fit(make_dataset(train_df, text_col, best_params["batch_size"], shuffle=True),
                  validation_data=make_dataset(dev_df, text_col, best_params["batch_size"]),
                  epochs=FINAL_EPOCHS, verbose=1,
                  callbacks=[tf.keras.callbacks.EarlyStopping(
                      monitor="val_loss", patience=3, restore_best_weights=True)])

        val_ds = make_dataset(dev_df, text_col, best_params["batch_size"])
        y_true, _, y_prob = predict(model, val_ds, dev_df)
        thr, micro = best_threshold(y_true, y_prob)

        key = f"{model_name}_{track_key}"
        thresholds[key] = thr
        best_configs[key] = best_params
        model.save(MODELS / f"{model_name.lower()}_{track_key}.keras")

        print(f"  saved {MODELS / f'{model_name.lower()}_{track_key}.keras'}")
        print(f"  threshold {thr:.2f} | dev Micro-F1 {micro * 100:.2f}%")

## 3.8 Persist everything the later stages need

The original notebook computed the tuned thresholds but never wrote them to disk,
so they were lost when the kernel ended — and without them the served model falls
back to 0.5 and stops predicting rare emotions almost entirely. They are saved
here alongside the models.

In [ ]:
(RESULTS / "thresholds.json").write_text(
    json.dumps(thresholds, indent=2), encoding="utf-8")
(RESULTS / "best_configs.json").write_text(
    json.dumps(best_configs, indent=2), encoding="utf-8")
pd.concat(tuning_log, ignore_index=True).to_csv(
    RESULTS / "hyperparameter_search.csv", index=False)

print("thresholds.json:")
print(json.dumps(thresholds, indent=2))
print("\nartefacts written:")
for p in sorted(MODELS.iterdir()):
    print(f"  models/{p.name:26s} {p.stat().st_size / 1e6:.1f} MB")
for name in ("thresholds.json", "best_configs.json", "hyperparameter_search.csv",
             "pipeline_fixture.json"):
    print(f"  results/{name}")

## Summary

Four trained models, their tuned decision thresholds, the winning configurations
and the full search log are now on disk, together with the two processed corpora.

`04_evaluation.ipynb` scores them, and the Streamlit UI in `app/` serves them —
both read straight from these directories, so no copying is required.